RAG 동작 구조
------------------------------------
1. 문장읽어 와서 텍스트를 추출
2. 텍스트를 분할(chunk)
3. 임베딩 -> pdf 파일을 통으로 주게 되면 token full 발생, 비용 多
4. vector db 저장 (임베딩된 데이터 저장소)
5. 질문: 임베딩
6. 유사도검색
7. 검색된결과를 context 로 llm 질문 

In [1]:
from openai import OpenAI
import openai
import os
api_key = os.getenv("OPENAI_API_KEY")
openai.api_key = api_key
client = OpenAI()

### pymupdf: pdf 에서 텍스트 추출
### faiss: vector db

In [2]:
%pip install pymupdf
%pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import fitz
import faiss
import numpy as np

In [4]:
my = []
my.extend(['dd','ee'])
# my.append('dd')
# my.append('ee')
my

['dd', 'ee']

In [5]:
[ n*10 for n in range(0,5,2)] #[0,2,4]

[0, 20, 40]

In [6]:
my =[]
my.extend( [ n*10 for n in range(0,5)] )
my.extend( [ n*20 for n in range(0,5)] )
my

[0, 10, 20, 30, 40, 0, 20, 40, 60, 80]

In [7]:
text_chunks = []
chunk_size=3
text = 'abcdefghijklmn'
text1 = 'opqrstuvwxyz'
text_chunks.extend( [ text[i:i+chunk_size] for i in range(0, len(text), chunk_size) ] )
text_chunks.extend( [ text1[i:i+chunk_size] for i in range(0, len(text1), chunk_size) ] )
text_chunks

['abc', 'def', 'ghi', 'jkl', 'mn', 'opq', 'rst', 'uvw', 'xyz']

### pdf 를 읽어서 chunk_size 텍스트를 분할

In [9]:
text_chunks = []
chunk_size=700
with fitz.open( '../pdf/The_Adventures_of_Tom_Sawyer.pdf') as pdf:
    for page in pdf:
        text = page.get_text()
        print( text)
        text_chunks.extend( [ text[i:i+chunk_size] for i in range(0, len(text), chunk_size) ] )
        print('-----------------')

 

-----------------
PENGUIN   READERS  2000 
 
 
 
 
 
 
 
www.penguinreaders.com

-----------------
 
 
The Adventures of                 
Tom Sawyer 
 
MARK TWAIN 
Level 1 
 
Retold by Jacqueline Kehl                                                    
Series Editors: Andy Hopkins and Jocelyn Potter 

-----------------
Pearson Education Limited                                                                            
Edinburgh Gate, Harlow,                                                                               
Essex CM20 2JE, England                                                                              
and Associated Companies throughout the world. 
ISBN 0 582 41923 9 
 
First published 1876                                                                                  
Published by Puffin Books 1950                                                                         
This edition first published 2000 
Copyright © Penguin Books 2000 
Typeset by Digital Type, 

In [10]:
text_chunks

[' \n',
 'PENGUIN   READERS  2000 \n \n \n \n \n \n \n \nwww.penguinreaders.com\n',
 ' \n \nThe Adventures of                 \nTom Sawyer \n \nMARK TWAIN \nLevel 1 \n \nRetold by Jacqueline Kehl                                                    \nSeries Editors: Andy Hopkins and Jocelyn Potter \n',
 'Pearson Education Limited                                                                            \nEdinburgh Gate, Harlow,                                                                               \nEssex CM20 2JE, England                                                                              \nand Associated Companies throughout the world. \nISBN 0 582 41923 9 \n \nFirst published 1876                                                                                  \nPublished by Puffin Books 1950                                                                         \nThis edition first published 2000 \nCopyright © Penguin Books 2000 \nTypeset by Digital Type, London \nS

In [11]:
len(text_chunks )

60

### 문장을 embedding 한다

In [12]:
response= client.embeddings.create( input="톰소오여는 어디로 여행했니", model="text-embedding-ada-002")
print( response )
emb = response.data[0].embedding
print( len(emb))
print( emb )
np.array(emb)

CreateEmbeddingResponse(data=[Embedding(embedding=[0.0006094318814575672, -0.017273040488362312, 0.00041706699994392693, -0.0308845154941082, -0.018546629697084427, 0.03603193536400795, -0.021717334166169167, 0.00993000902235508, -0.0010074282763525844, 0.003930213861167431, 0.009651411324739456, -0.0036317165940999985, -0.011555160395801067, -0.022340860217809677, 0.011740892194211483, 0.002623459091410041, 0.02188979834318161, 0.0006173088913783431, 0.010752534493803978, -0.019316088408231735, -0.00246426067315042, -0.01581372134387493, -0.008404356427490711, 0.009233514778316021, -0.004308310337364674, 0.01761797070503235, 0.03099064901471138, -0.027939341962337494, -0.01018870621919632, 0.003333219327032566, 0.03268876671791077, 0.01177405845373869, -0.024304309859871864, -0.027859743684530258, -0.01954161934554577, -0.013074180111289024, -0.007077701389789581, -0.004925204440951347, 0.000723026692867279, -0.015415724366903305, -0.010566802695393562, 0.005920195486396551, 0.0009866

array([ 0.00060943, -0.01727304,  0.00041707, ...,  0.0068588 ,
        0.005562  ,  0.00467646], shape=(1536,))

In [13]:
def get_embedding( text):
    response= client.embeddings.create( input=text,model="text-embedding-ada-002")
    return np.array( response.data[0].embedding , dtype=np.float32)

### faiss db에 임베딩을 저장하기

In [14]:
np.vstack( [[1,2],[3,4], [5,6]])

array([[1, 2],
       [3, 4],
       [5, 6]])

In [15]:
def store_embedding_in_faiss( text_chunks ):
    embeddings = np.vstack( [ get_embedding(chunk) for chunk in text_chunks ] )
    print( embeddings.shape ) # (60, 1536)
    index = faiss.IndexFlatL2( embeddings.shape[1] ) # IndexFlatL2 :  유클리드 거리 -> 두 점(벡터) 사이의 "직선 거리"
    index.add( embeddings ) # embedding 벡터 db 저장
    return index

In [16]:
index = store_embedding_in_faiss(text_chunks)

(60, 1536)


In [17]:
" ".join( ['aa','bb','cc'] )

'aa bb cc'

In [18]:
question = "Tell me about the characters"
q_emb = get_embedding( question ).reshape( 1,-1)
_, top_index = index.search( q_emb , k=5 )
print( top_index)
" ".join( [text_chunks[i] for i in top_index[0] ] )

[[51 23 55 29 58]]


'ACTIVITIES \n \nChapters 1-6                                                                                 \nBefore you read \n1 Find the words in italics in your dictionary. They are all in the \nstory. \na   Answer the questions. \n    What adventures are on TV? What adventures do   you \nhave? \nWhat are you afraid of? \nDo you like cats? \nWhat makes you sac/? \n    When are you surprised?                                                            \nb    Put a word on the left with a word on the right. \naunt                            dead \nchurch                             family \nfence \nsick \ngraveyard \npolice \nmedicine \npicture \npaint \nyard \ntrial \nSunday \nc    Put these words in the sentences. ng, Joe and Huck said, “We’re not happy \nhere now. We want to go home.” \nTom said, “Let’s go home on Sunday. We can go to \nchurch. People are going to be very surprised!” \nSunday morning, many children were at church. They \ntalked about the three boys. They were sad 

In [19]:
text_chunks[51]

'ACTIVITIES \n \nChapters 1-6                                                                                 \nBefore you read \n1 Find the words in italics in your dictionary. They are all in the \nstory. \na   Answer the questions. \n    What adventures are on TV? What adventures do   you \nhave? \nWhat are you afraid of? \nDo you like cats? \nWhat makes you sac/? \n    When are you surprised?                                                            \nb    Put a word on the left with a word on the right. \naunt                            dead \nchurch                             family \nfence \nsick \ngraveyard \npolice \nmedicine \npicture \npaint \nyard \ntrial \nSunday \nc    Put these words in the sentences.'